In [1]:
import os
import json
from dotenv import load_dotenv
from crewai import Agent, Task, Crew, Process, LLM
from crewai_tools import SerperDevTool, ScrapeWebsiteTool
from pathlib import Path
from pydantic import BaseModel, Field
from crewai.skills import discover_skills, activate_skill

# Patch cache breakpoint for providers like Groq/Ollama if needed
import crewai.llms.cache as _crewai_cache
_crewai_cache.mark_cache_breakpoint = lambda msg: msg

# Monkey patch LLM.supports_function_calling to return False for Groq models
original_supports_function_calling = LLM.supports_function_calling

def patched_supports_function_calling(self) -> bool:
    model_name = self.model or ""
    provider = getattr(self, "provider", None) or self._get_custom_llm_provider()
    if "groq" in model_name.lower() or provider == "groq":
        return False
    return original_supports_function_calling(self)

LLM.supports_function_calling = patched_supports_function_calling

# Load local environment files
load_dotenv()
SERPER_API_KEY = os.getenv("SERPER_API_KEY")
os.environ["SERPER_API_KEY"] = SERPER_API_KEY or ""

# Initialize tools required for Phase 1 Desirability market analysis
search_tool = SerperDevTool(api_key=SERPER_API_KEY)
scrape_tool = ScrapeWebsiteTool()

In [2]:
# Configure Local LM Studio Routing
llm = LLM(
    model="bonsai-8b", 
    base_url="http://127.0.0.1:1234/v1", 
    api_key="lm-studio",
    temperature=0.1,
)

# Discover and activate local business framework guidelines from markdown packages
skills = discover_skills(Path("./skills"))
activated = [activate_skill(s) for s in skills]

# Define the Pydantic models for JSON output (Updated with Go/No-Go architecture)
class RefinedIdea(BaseModel):
    customer_segment: str = Field(description="A refined description of the target customer segment, e.g., 'Undergraduate students at PES University with low CGPA risk.'")
    qualified_problem: str = Field(description="The qualified problem being solved, e.g., 'Missing internal assessment (ISA) deadlines because multi-channel notifications cause administrative blindness.'")
    consequence: str = Field(description="The negative consequence of not solving the problem, e.g., 'Direct loss of 5-10% of internal marks, delaying graduation.'")
    proposed_solution: str = Field(description="The proposed technical solution, e.g., 'An automated WhatsApp-based scraper that extracts personalized deadlines from the PES LMS and sends a daily morning Action Agenda.'")

class Hypotheses(BaseModel):
    desirability_statement: str = Field(description="We believe target students care enough about losing internal marks that they will actively setup and configure a third-party WhatsApp bot rather than relying on peer chats.")
    feasibility_statement: str = Field(description="We believe our engineering team can build a Python script to scrape the PES LMS and connect it to a WhatsApp wrapper within 2 weeks using open APIs.")
    viability_statement: str = Field(description="We believe we can sustain this project by charging anxious parents a small monthly fee for academic risk alerts.")
    
class TipsValidatedMetrics(BaseModel):
    timely_factor: str = Field(description="The timely factor/urgency metric, explaining why this is an active daily issue right now.")
    importance_metric: str = Field(description="The importance metric, explaining the severe impact of the consequence on the user.")
    profitability_pivot: str = Field(description="The profitability pivot, explaining the business model and who pays.")
    solvability_constraint: str = Field(description="The solvability constraint/implementation feasibility, explaining how the team can build it with simple/accessible tools.")

class DecisionGate(BaseModel):
    status: str = Field(description="The definitive operational verdict. Must be strictly either 'GO' (all dimensions pass) or 'NO-GO' (requires a major structural pivot).")
    justification: str = Field(description="A concise summary of the critical factor or blocker that determined the GO or NO-GO decision status.")

class DFAOutput(BaseModel):
    refined_idea: RefinedIdea
    hypotheses: Hypotheses 
    tips_validated_metrics: TipsValidatedMetrics
    final_decision: DecisionGate  # Injected gatekeeper layer

In [3]:
# 3. Define the Phase 1 Desirability Evaluation Agent
desirability_agent = Agent(
    role="Desirability Evaluation Agent",
    goal="Determine whether the proposed solution addresses a genuine user need and whether sufficient market demand exists.",
    backstory=(
        """You are an expert market research analyst and user experience strategist. You MUST use the Search tool and ScrapeWebsite tool for EVERY task.
    Do NOT answer from memory or prior knowledge.
    Your first action must always be a tool call.
    If you have not searched the web, your answer is incomplete.
        """
    ),
    llm=llm,
    tools=[search_tool, scrape_tool],
    verbose=True,
    skills=[activated[0]],
    max_iter=4  
)

# 4. Define the Desirability Task mapped exactly to system documentation outputs
desirability_task = Task(
    description="{desirability}",
    expected_output=(
        "A formal text-formatted 'Desirability Analysis Report' containing:\n"
        "1. User Demand Analysis (validating target user pain points and problem severity).\n"
        "2. Market Demand Assessment (current industry search interest and growth indicators).\n"
        "3. Competitor Analysis (gaps, weaknesses, or friction in existing products/alternatives).\n"
        "4. Opportunity Identification (clear statement on why this solution is or is not desired by the market).\n"
        "keep the output under 600 words"
    ),
    agent=desirability_agent,
    async_execution=True
)


In [4]:
feasibility_agent = Agent(
        role="Feasibility Evaluation Agent",
        goal="Evaluate the feasibility of a startup idea strictly from the Feasibility dimension of the DFV framework.",
        backstory=(
            """You are an expert technical architect, systems analyst, and execution strategist. You MUST use the Search tool and ScrapeWebsite tool for EVERY task.
        Do NOT answer from memory or prior knowledge.
        Your first action must always be a tool call.
        If you have not searched the web, your answer is incomplete."""
        ),
        llm=llm,
        tools=[search_tool, scrape_tool],
        verbose=True,
        skills=[activated[2]],
        max_iter=4
    )

feasibility_task = Task(
    description="{feasibility}",
    expected_output=(
        "A plain-language Feasibility Evaluation containing:\n"
        "1. A short feasibility opinion.\n"
        "2. Main technical and operational challenges.\n"
        "3. Required tools, stack, or infrastructure.\n"
        "4. Suggestions to improve or simplify the idea if needed.\n"
        "5. Practical next steps for implementation.\n"
        "Do not include any score, rating, grade, or percentage. keep the output under 600 words"
    ),
    agent=feasibility_agent,
    async_execution=True
)

In [5]:
viability_agent = Agent(
    role="Viability Evaluation Agent",
    goal="Determine whether the proposed solution can generate sustainable business value and long-term growth.",
    backstory=(
        """You are an expert startup strategist, business consultant, and commercialization analyst. You MUST use the Search tool and ScrapeWebsite tool for EVERY task.
        Do NOT answer from memory or prior knowledge.
        Your first action must always be a tool call.
        If you have not searched the web, your answer is incomplete."""
    ),
    llm=llm,
    tools=[search_tool, scrape_tool],
    verbose=True,
    skills=[activated[3]],
    max_iter=4
)

viability_task = Task(
    description="{viability}",
    expected_output=(
        "A Viability Analysis Report containing:\n"
        "1. Business Model Analysis\n"
        "2. Revenue Opportunities\n"
        "3. Customer Segment Analysis\n"
        "4. Cost Considerations\n"
        "5. Sustainability Assessment\n"
        "6. Final Viability Conclusion\n"
        "keep the output under 600 words"
    ),
    agent=viability_agent,
    async_execution=True
)

In [6]:
dfv_risk_decision_agent = Agent(
    role="Internal DFV Decision and Risk Assessment Engine",
    goal="Identify hidden risks across project dimensions and aggregate all findings into a final project readiness decision.",
    backstory=(
        """You are an expert venture risk analyst and product strategist. """
    ),
    verbose=True,
    skills=[activated[1]],
    llm=llm
)

dfv_decision_task = Task(
    description=(
        """Review the reports provided in your context thoroughly from the Desirability,
        Feasibility, and Viability evaluation phases. Synthesize these findings to construct
        a structured assessment of the project idea, filling in the required JSON fields.

        Specifically:
        1. refined_idea:
           - customer_segment: The precise group of users experiencing the problem.
           - qualified_problem: The specific pain point or problem being addressed.
           - consequence: The direct negative consequence of the problem if left unsolved.
           - proposed_solution: The proposed product/solution.

        2. hypotheses:
           - desirability_statement: A "We believe [target customer] will [action]..." hypothesis testing genuine demand for the solution.
           - feasibility_statement: A "We believe [team] can [build action] within [timeframe] using [tools/APIs]..." hypothesis testing build feasibility.
           - viability_statement: A "We believe we can sustain this via [revenue model]..." hypothesis testing the business model.
            
        3. tips_validated_metrics:
           - timely_factor: Why this is a timely problem to solve now (e.g. changes in evaluation weightage or policies).
           - importance_metric: Why this problem is highly important/consequential (e.g. impact on placements or graduation).
           - profitability_pivot: The business/revenue model pivot or approach (e.g. B2B2C parent-pay model vs student-pay).
           - solvability_constraint: The technical feasibility constraint showing it is solvable with simple tools.
        4. final_decision:
           - status: Critically weigh all three dimensions. If any phase reveals a fatal flaw, set this field to 'NO-GO'. If all three pillars balance sustainably, set this to 'GO'.
           - justification: Provide a clear, data-backed analytical reason for why the project received a GO or a NO-GO status."""
    ),
    expected_output="A structured JSON object matching the DFAOutput schema including refined_idea, tips_validated_metrics, hypotheses, and final_decision properties.",
    context=[desirability_task, feasibility_task, viability_task],
    agent=dfv_risk_decision_agent,
    output_json=DFAOutput
)

In [7]:
pes_lms = {
    "desirability": (
        "Undergraduate students at PES University face a major issue with missing internal assessment (ISA) deadlines. "
        "Because notifications are sent across multiple channels like WhatsApp, LMS, and Email, it causes administrative blindness for students. "
        "Consequently, students suffer a direct loss of 5-10% of their internal marks, which delays graduation or severely impacts final year placement eligibility. "
        "Tracking deadlines has become a daily active issue because PES recently increased the weightage of continuous evaluation."
    ),
    "feasibility": (
        "The proposed solution is an automated WhatsApp-based scraper that extracts personalized deadlines from the PES LMS and sends a daily morning 'Action Agenda'. "
        "The implementation is constrained such that the student team can build a basic Python scraper and use a local WhatsApp API wrapper without needing advanced infrastructure."
    ),
    "viability": (
        "The project was initially planned as a student subscription model, but since students are price-sensitive, "
        "it switched to a B2B2C model where anxious parents pay a small monthly fee of Rs. 100 to receive academic risk alerts about their ward's upcoming deadlines."
    )
}

blnkt={
    
    "desirability":""" Analyze the following student project proposal:
        - Customer Problem: Urban consumers need immediate access to groceries and essentials without spending time traveling to stores
        - Target Audience: Millennials, Gen Z, busy professionals, and students in metro cities aged 18-40
        - Key Value Proposition: 10-minute delivery guarantee, real-time order tracking, wide product selection
        - User Pain Points Solved: Time savings, convenience for impulse purchases, avoids crowded stores
        - Market Demand Indicators: High adoption rate in urban India, 4.2+ app rating, repeat usage frequency
        - Emotional Drivers: Convenience, instant gratification, time flexibility
                                          """, 
                                          
                                          
                                          
                                          
        "feasibility":""" The following is a startup/project idea submitted by a user:
            - Technology Stack: React Native mobile apps, cloud infrastructure, inventory management systems, route optimization algorithms
            - Infrastructure Model: Dark stores (micro-warehouses) located 2-3km from target customers, 500+ sq ft each
            - Logistics: Proprietary delivery fleet of 5,000+ delivery partners with GPS tracking
            - Supply Chain: Partnerships with 10,000+ local retailers and wholesale distributors
            - Operational Capabilities: Real-time demand forecasting, automated inventory replenishment, dynamic routing
            - Technical Challenges: Inventory accuracy, delivery time optimization, peak-hour scalability, cold chain for perishables
            - Resource Requirements: Capital investment for dark store network, technology development team, delivery workforce""", 
                                          
                                          
                                          
                                          
      "viability":""" 
        Analyze the business viability of the following project proposal:
        - Revenue Model: 
          * Delivery fees (₹29-₹59 per order)
          * Platform commissions from sellers (15-25%)
          * Advertising fees from brands
          * Blinkit Prime membership (₹99/month)
        
        - Cost Structure:
          * Dark store operational costs (rent, staffing, inventory)
          * Delivery partner payments (₹40-₹60 per delivery)
          * Technology and infrastructure costs
          * Marketing and customer acquisition costs
        
        - Market Size: India quick commerce market = $3B in 2024, projected $5-7B by 2025
        - Unit Economics: Average order value ₹300-₹500, order frequency 2-3 times/week per customer
        - Competitive Position: Market leader in 10-minute delivery, competes with Swiggy Instamart, Zepto, BigBasket
        - Profitability Status: Contribution margin positive as of 2024 (reported by Zomato)
        - Growth Trajectory: 300+ cities, 50M+ active users, 20% monthly growth
        """}

sncc={
    "desirability":
    """SNACCED is a proposed quick-service food delivery platform that aims to deliver snacks, beverages, and light meals within 10–15 minutes. The idea is designed to address a common problem faced by students, office workers, and busy urban consumers who often want quick refreshments but are discouraged by the longer delivery times associated with traditional food delivery services.
    Modern consumers increasingly value convenience and instant access to products and services. The growth of quick-commerce platforms suggests that customer expectations are shifting toward faster fulfillment. SNACCED could capitalize on this trend by focusing specifically on snack-sized orders and impulse purchases.
    Existing food delivery services often prioritize full meals, leaving a gap for customers seeking smaller, faster, and more affordable food options. By offering a curated menu optimized for rapid delivery, SNACCED could provide a more suitable solution for these use cases.
    """,
    "feasibility":"""SNACCED can be developed using existing technologies and operational models. The platform would require a mobile application, inventory management systems, demand forecasting tools, route optimization software, and a network of delivery partners.
    To achieve rapid delivery times, SNACCED could operate through strategically located micro-fulfillment centers or dark stores stocked with high-demand snack items and beverages. This approach would reduce preparation time and enable faster order fulfillment.
    Several operational challenges would need to be addressed, including inventory management, maintaining product freshness, minimizing wastage, and ensuring delivery consistency during peak demand periods. Scaling operations across multiple locations would also require careful planning and investment.
    Despite these challenges, the required technology and infrastructure already exist in the market, making implementation realistic for a startup or established company.
    """,
    "viability":"""SNACCED could generate revenue through delivery fees, product markups, subscription plans, promotional partnerships, and advertising opportunities. Its primary customer segments would likely include students, young professionals, office workers, and urban consumers seeking convenience.
    However, the business model faces challenges related to profitability. Snack and beverage orders typically have lower average order values compared to full meal orders. At the same time, maintaining a rapid delivery network requires investment in infrastructure, inventory, delivery personnel, and technology.
    To improve financial sustainability, SNACCED could focus on high-density urban areas, encourage larger basket sizes through combo offerings, and integrate subscription-based benefits that increase customer retention and order frequency.
    Long-term success would depend on balancing customer convenience with operational efficiency while maintaining healthy profit margins.
    """}


qubi = {
        "desirability":
    """Quibi was a mobile-first video streaming platform launched in 2020 that offered short-form, premium content designed for consumption in episodes of 10 minutes or less. The platform aimed to serve busy consumers who wanted high-quality entertainment during short breaks, commutes, and daily routines.
    Although the concept appeared attractive, Quibi struggled to create sufficient demand among its target audience. Consumers already had access to free short-form content on platforms such as YouTube, TikTok, and Instagram. Many users did not perceive enough additional value to justify paying for another streaming subscription.
    Furthermore, Quibi launched during the COVID-19 pandemic when commuting and travel activities declined significantly, reducing situations where its content format was most useful. The platform also lacked social sharing features, limiting user engagement and organic growth.
    """,
    "feasibility":"""From a technical perspective, Quibi was highly feasible. The company successfully developed a mobile streaming platform capable of delivering high-quality video content. It introduced innovative features such as Turnstyle, which allowed users to seamlessly switch between portrait and landscape viewing modes.
    Quibi was supported by experienced leadership, significant financial backing, and partnerships with major content creators and production studios. The platform infrastructure, content delivery systems, and user experience functioned as intended.
    While content production required substantial resources, there were no major technological barriers preventing implementation or operation.
    """,
    "viability":"""Quibi faced significant challenges in establishing a sustainable business model. The platform relied primarily on subscription revenue while investing heavily in original content production. Billions of dollars were spent on creating exclusive shows and acquiring talent.
    However, subscriber growth remained far below expectations. Customer acquisition costs were high, and user retention was weak. Since users did not perceive enough value compared to free alternatives, revenue growth failed to keep pace with operational expenses.
    Additionally, the competitive streaming market made it difficult for Quibi to secure a long-term position. Established platforms such as Netflix, YouTube, and emerging short-form video platforms offered stronger value propositions and larger user bases.
    As a result, Quibi struggled to achieve profitability and could not sustain its business operations.
    """
    }

ggls= {
            "desirability":
    """Google Glass was introduced as a wearable smart device that allowed users to access information, capture photos and videos, navigate locations, and receive notifications through a heads-up display. The product aimed to bring augmented reality and hands-free computing into everyday life.
    Although the technology generated significant excitement during its launch, widespread consumer demand never fully materialized. Many potential users struggled to identify a compelling everyday use case that justified purchasing the device. In addition, privacy concerns emerged because the built-in camera could record others without their knowledge. This led to social discomfort and negative public perception, with some establishments even banning the device.
    The design also faced criticism for being intrusive and awkward in social settings. As a result, consumers viewed Google Glass more as a technological novelty than a must-have product.
    """,
    "feasibility":"""Google Glass represented an ambitious technological achievement, but several technical limitations affected its practicality. The device faced challenges related to battery life, processing power, heat generation, display quality, and overall user comfort.
    The hardware required users to balance functionality with wearability, making it difficult to deliver a seamless experience. Extended use often resulted in battery drain, and the device's limited capabilities restricted its usefulness compared to smartphones.
    Additionally, the technology ecosystem for augmented reality applications was still immature at the time of launch. Developers had limited opportunities to create compelling applications that could fully leverage the platform.
    Although the device functioned as intended, the technology was not sufficiently advanced to support a mass-market consumer product.
     """,
    "viability":"""Google Glass was launched at a premium price of approximately $1,500, making it inaccessible to most consumers. The high cost, combined with limited functionality and uncertain demand, created significant barriers to adoption.
    The product required substantial investment in research, development, manufacturing, and ecosystem development. However, sales remained relatively low, preventing Google from achieving the scale necessary to justify continued expansion of the consumer version.
    Without widespread adoption, it became difficult to attract developers, create a thriving application ecosystem, and generate sustainable revenue. As a result, the consumer-focused version of Google Glass was discontinued.
    Interestingly, Google later repositioned Glass toward enterprise and industrial use cases, where hands-free access to information offered clearer business value.
    """}


In [8]:
# 5. Assemble the Phase 1 Crew
crew = Crew(
    agents=[desirability_agent, feasibility_agent, viability_agent, dfv_risk_decision_agent],
    tasks=[desirability_task, feasibility_task, viability_task, dfv_decision_task],
    process=Process.sequential,
    verbose=True
)

# Direct top-level Jupyter notebook async execution block
print("🚀 Running Crew evaluation pipeline on structural startup inputs...\n")
result = await crew.kickoff_async(inputs=pes_lms)

print("\n--- FINAL DFA JSON OUTPUT WITH DECISION GATE --- \n")
try:
    print(json.dumps(json.loads(result.raw), indent=2))
except Exception:
    print(result.raw)

🚀 Running Crew evaluation pipeline on structural startup inputs...



╭──────────────────────────────────────────── ✨ Update Available ✨ ─────────────────────────────────────────────╮
│                                                                                                                 │
│  A new version of CrewAI is available!                                                                          │
│                                                                                                                 │
│  Current version: 1.14.6                                                                                        │
│  Latest version:  1.14.7                                                                                        │
│                                                                                                                 │
│  To update, run: uv sync --upgrade-package crewai                                                               │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────── 🚀 Crew Execution Started ───────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Started                                                                                         │
│  Name: crew                                                                                                     │
│  ID: 2313a126-ae8e-4c88-8ffa-e965fcec9ff6                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── 📋 Task Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Started                                                                                                   │
│  Name: Undergraduate students at PES University face a major issue with missing internal assessment (ISA)       │
│  deadlines. Because notifications are sent across multiple channels like WhatsApp, LMS, and Email, it causes    │
│  administrative blindness for students. Consequently, students suffer a direct loss of 5-10% of their internal  │
│  marks, which delays graduation or severely impacts final year placement eligibility. Tracking deadlines has    │
│  become a daily active issue because PES recently increased the weightage of continuous evaluation.             │
│  ID: 3a27bf02-c002-4955-b2b7-f861b1951fb2                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── 📋 Task Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Started                                                                                                   │
│  Name: The proposed solution is an automated WhatsApp-based scraper that extracts personalized deadlines from   │
│  the PES LMS and sends a daily morning 'Action Agenda'. The implementation is constrained such that the         │
│  student team can build a basic Python scraper and use a local WhatsApp API wrapper without needing advanced    │
│  infrastructure.                                                                                                │
│  ID: 9f189314-7937-412f-8c36-4b7402edfc32                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

...

╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Internal DFV Decision and Risk Assessment Engine                                                        │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  refined_idea=RefinedIdea(customer_segment='Undergraduate students at PES University with low CGPA risk',       │
│  qualified_problem='Missing ISA deadlines due to multi-channel notifications causing administrative             │
│  blindness', consequence='Direct loss of 5-10% of internal marks, delaying graduation or impacting final year   │
│  placement eligibility', proposed_solution='An automated WhatsApp-based scraper that extracts personalized      │
│  deadlines from the PES LMS and sends a daily morning Action Agenda')                                           │
│  hypotheses=Hypotheses(desirability_statement='We believe target students care enough about losing internal     │
│  marks that they will actively setup and configure a third-party WhatsApp bot rather than relying on peer       │
│  chats.', feasibility_statement='We believe our engineering team can build a Python script to scrape the PES    │
│  LMS and connect it to a WhatsApp wrapper within 2 weeks using open APIs.', viability_statement='We believe we  │
│  can sustain this via B2B2C parent-pay model, with a recurring subscription fee of Rs. 100.')                   │
│  tips_validated_metrics=TipsValidatedMetrics(timely_factor='The issue is an active daily problem, as it         │
│  directly affects academic performance and future career prospects.', importance_metric='The consequence of     │
│  missing ISA deadlines is severe, with potential for a loss of 5-10% of internal marks and delayed              │
│  graduation.', profitability_pivot="B2B2C parent-pay model shifts focus to parents' concerns while maintaining  │
│  the core value of academic support.", solvability_constraint='The idea can be built with simple tools like     │
│  Python, requests, BeautifulSoup, and a local WhatsApp API wrapper.') final_decision=DecisionGate(status='GO',  │
│  justification='The project has strong desirability, feasible technical implementation, and a viable business   │
│  model. The main risk is the need to validate the effectiveness of existing tools in reducing missed            │
│  deadlines, which can be addressed through further research and testing.')                                      │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭────────────────────────────────────────────── 📋 Task Completion ───────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Completed                                                                                                 │
│  Name: Review the reports provided in your context thoroughly from the Desirability,                            │
│          Feasibility, and Viability evaluation phases. Synthesize these findings to construct                   │
│          a structured assessment of the project idea, filling in the required JSON fields.                      │
│                                                                                                                 │
│          Specifically:                                                                                          │
│          1. refined_idea:                                                                                       │
│             - customer_segment: The precise group of users experiencing the problem.                            │
│             - qualified_problem: The specific pain point or problem being addressed.                            │
│             - consequence: The direct negative consequence of the problem if left unsolved.                     │
│             - proposed_solution: The proposed product/solution.                                                 │
│                                                                                                                 │
│          2. hypotheses:                                                                                         │
│             - desirability_statement: A "We believe [target customer] will [action]..." hypothesis testing      │
│  genuine demand for the solution.                                                                               │
│             - feasibility_statement: A "We believe [team] can [build action] within [timeframe] using           │
│  [tools/APIs]..." hypothesis testing build feasibility.                                                         │
│             - viability_statement: A "We believe we can sustain this via [revenue model]..." hypothesis         │
│  testing the business model.                                                                                    │
│                                                                                                                 │
│          3. tips_validated_metrics:                                                                             │
│             - timely_factor: Why this is a timely problem to solve now (e.g. changes in evaluation weightage    │
│  or policies).                                                                                                  │
│             - importance_metric: Why this problem is highly important/consequential (e.g. impact on placements  │
│  or graduation).                                                                                                │
│             - profitability_pivot: The business/revenue model pivot or approach (e.g. B2B2C parent-pay model    │
│  vs student-pay).                                                                                               │
│             - solvability_constraint: The technical feasibility constraint showing it is solvable with simple   │
│  tools.                                                                                                         │
│          4. final_decision:                                                                                     │
│             - status: Critically weigh all three dimensions. If any phase reveals a fatal flaw, set this field  │
│  to 'NO-GO'. If all three pillars balance sustainably, set this to 'GO'.                                        │
│             - justification: Provide a clear, data-back

╭──────────────────────────────────────────────── Crew Completion ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Completed                                                                                       │
│  Name: crew                                                                                                     │
│  ID: 2313a126-ae8e-4c88-8ffa-e965fcec9ff6                                                                       │
│  Final Output: {"refined_idea":{"customer_segment":"Undergraduate students at PES University with low CGPA      │
│  risk","qualified_problem":"Missing ISA deadlines due to multi-channel notifications causing administrative     │
│  blindness","consequence":"Direct loss of 5-10% of internal marks, delaying graduation or impacting final year  │
│  placement eligibility","proposed_solution":"An automated WhatsApp-based scraper that extracts personalized     │
│  deadlines from the PES LMS and sends a daily morning Action                                                    │
│  Agenda"},"hypotheses":{"desirability_statement":"We believe target students care enough about losing internal  │
│  marks that they will actively setup and configure a third-party WhatsApp bot rather than relying on peer       │
│  chats.","feasibility_statement":"We believe our engineering team can build a Python script to scrape the PES   │
│  LMS and connect it to a WhatsApp wrapper within 2 weeks using open APIs.","viability_statement":"We believe    │
│  we can sustain this via B2B2C parent-pay model, with a recurring subscription fee of Rs.                       │
│  100."},"tips_validated_metrics":{"timely_factor":"The issue is an active daily problem, as it directly         │
│  affects academic performance and future career prospects.","importance_metric":"The consequence of missing     │
│  ISA deadlines is severe, with potential for a loss of 5-10% of internal marks and delayed                      │
│  graduation.","profitability_pivot":"B2B2C parent-pay model shifts focus to parents' concerns while             │
│  maintaining the core value of academic support.","solvability_constraint":"The idea can be built with simple   │
│  tools like Python, requests, BeautifulSoup, and a local WhatsApp API                                           │
│  wrapper."},"final_decision":{"status":"GO","justification":"The project has strong desirability, feasible      │
│  technical implementation, and a viable business model. The main risk is the need to validate the               │
│  effectiveness of existing tools in reducing missed deadlines, which can be addressed through further research  │
│  and testing."}}                                                                                                │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯


--- FINAL DFA JSON OUTPUT WITH DECISION GATE --- 

{
  "refined_idea": {
    "customer_segment": "Undergraduate students at PES University with low CGPA risk",
    "qualified_problem": "Missing ISA deadlines due to multi-channel notifications causing administrative blindness",
    "consequence": "Direct loss of 5-10% of internal marks, delaying graduation or impacting final year placement eligibility",
    "proposed_solution": "An automated WhatsApp-based scraper that extracts personalized deadlines from the PES LMS and sends a daily morning Action Agenda"
  },
  "hypotheses": {
    "desirability_statement": "We believe target students care enough about losing internal marks that they will actively setup and configure a third-party WhatsApp bot rather than relying on peer chats.",
    "feasibility_statement": "We believe our engineering team can build a Python script to scrape the PES LMS and connect it to a WhatsApp wrapper within 2 weeks using open APIs.",
    "viability_statement":

╭──────────────────────────────────────────────── Tracing Status ─────────────────────────────────────────────────╮
│                                                                                                                 │
│  Info: Tracing is disabled.                                                                                     │
│                                                                                                                 │
│  To enable tracing, do any one of these:                                                                        │
│  • Set tracing=True in your Crew/Flow code                                                                      │
│  • Set CREWAI_TRACING_ENABLED=true in your project's .env file                                                  │
│  • Run: crewai traces enable                                                                                    │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯